<a href="https://colab.research.google.com/github/IIskel/RAG/blob/main/Building_An_agentic_RAG_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#import local file
from google.colab import files
uploaded = files.upload()

Saving Machine Learning with Python Cookbook (en).pdf to Machine Learning with Python Cookbook (en).pdf


In [3]:
#List the file
import os
current_path = os.getcwd()
for x in os.listdir(current_path):
  print(x)

.config
Machine Learning with Python Cookbook (en).pdf
sample_data


### 1. Setup and Dependency Installation

First, we'll install all necessary Python packages. These packages include `langchain`, `langchain-community`, `langchain-chroma`, `transformers`, `sentence-transformers`, and `pypdf`, which are crucial for building our RAG (Retrieval-Augmented Generation) system.

In [5]:
%%shell
# Create a requirements.txt file with all necessary libraries
echo "langchain
langchain-community
langchain-chroma
transformers
sentence-transformers
pypdf" > requirements.txt

# Install packages from requirements.txt
pip install -r requirements.txt

# Uninstall and reinstall langchain-chroma and chromadb to ensure compatibility
# This step resolves potential dependency conflicts and ensures the latest compatible versions.
pip uninstall -y chromadb langchain-chroma
pip install langchain-chroma

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 

### 2. Import Libraries and Load PDF Documents

We import the required libraries for document loading, text splitting, embeddings, and the RAG framework. Then, we define a function to load PDF documents from a specified folder. In this case, we'll load the `Machine Learning with Python Cookbook (en).pdf` file, which was previously uploaded to the Colab environment.

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from transformers import pipeline
from langchain_chroma import Chroma

# Function to load PDF documents from a specified folder
def load_docs(folder_path):
    docs = []
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            loader = PyPDFLoader(os.path.join(folder_path, file))
            docs.extend(loader.load())
    return docs

# Specify the folder where your PDFs are stored (e.g., '/content' for Colab uploads)
docs = load_docs("/content")
print(f"PDF Pages Loaded: {len(docs)}")

PDF Pages Loaded: 427


### 3. Split Documents into Chunks

To effectively process large documents and retrieve relevant information, we split the loaded PDF pages into smaller, manageable chunks. We use `RecursiveCharacterTextSplitter` to ensure chunks are contextually coherent, with a defined `chunk_size` and `chunk_overlap`.

In [7]:
# Initialize the text splitter with a chunk size of 500 characters and an overlap of 80 characters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80
)

# Split the documents into chunks
chunks = text_splitter.split_documents(docs)
print(f"Chunks Created: {len(chunks)}")

Chunks Created: 1398


### 4. Create Embeddings and Vector Database

We convert the text chunks into numerical vector representations (embeddings) using a pre-trained `HuggingFaceEmbeddings` model (`all-MiniLM-L6-v2`). These embeddings are then stored in a `Chroma` vector database, which allows for efficient similarity search to retrieve relevant chunks.

In [8]:
# Initialize the HuggingFace embeddings model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Extract text content from chunks for embedding
texts = [c.page_content for c in chunks]

# Create a Chroma vector database and add the text embeddings
db = Chroma(
    collection_name="rag_store",
    embedding_function=embedding_model
)
db.add_texts(texts)

# Configure the retriever to fetch the top 3 most relevant documents
retriever = db.as_retriever(search_kwargs={"k": 3})
print("Chroma DB and Retriever initialized.")

/tmp/ipykernel_2389/2938124530.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Chroma DB and Retriever initialized.


### 5. Load the Local Large Language Model (LLM)

We load a local Large Language Model (`google/flan-t5-base`) using the `transformers` pipeline for text generation. This LLM will be used to generate answers based on the retrieved context.

In [9]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the tokenizer and model for Flan-T5
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def custom_llm_generate(prompt, max_new_tokens=500):
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

llm = custom_llm_generate # Replace the pipeline object with our custom generation function
print("Local LLM ('google/flan-t5-base') loaded with direct model usage.")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Local LLM ('google/flan-t5-base') loaded with direct model usage.


### 6. Define Agent Controller

This simple agent controller decides whether to perform a document search (RAG) or answer a query directly based on keywords present in the user's question. This helps in routing questions appropriately.

In [10]:
# Define an agent controller function to decide the action based on the query
def agent_controller(query):
    q = query.lower()
    # If the query contains keywords related to document content, trigger a search
    if any(word in q for word in ["pdf", "document", "data", "summarize", "information", "find", "book"]):
        return "search"
    # Otherwise, answer directly using the LLM without RAG
    return "direct"

print("Agent controller defined.")

Agent controller defined.


### 7. Implement the RAG Pipeline

The `rag_answer` function orchestrates the RAG process. It uses the `agent_controller` to determine if a search is needed. If so, it retrieves relevant document chunks from the Chroma DB and constructs a prompt with this context for the LLM. Otherwise, it sends the query directly to the LLM.

In [16]:
# Define the RAG answer generation function
def rag_answer(query):
    # Determine the action (search or direct) using the agent controller
    action = agent_controller(query)

    if action == "search":
        print(f"🕵️ Agent decided to SEARCH document for: '{query}'")
        # Retrieve relevant documents based on the query
        results = retriever.invoke(query)
        # Combine the page content of the retrieved documents into a single context string
        context = "\n".join([r.page_content for r in results])
        # Construct the final prompt for the LLM, including the retrieved context
        final_prompt = f"Use this context:\n{context}\n\nAnswer:\n{query}"
    else:
        print(f"🤖 Agent decided to answer DIRECTLY: '{query}'")
        # If no search is needed, the final prompt is just the original query
        final_prompt = query

    # Generate the response using the loaded LLM (custom_llm_generate)
    # The custom_llm_generate function returns a string directly.
    response = llm(final_prompt)
    return response

In [17]:

# --- Test the RAG Pipeline ---

print("\n--- Testing RAG Pipeline ---")

# Test 1: A document-specific question (should trigger RAG search)
query_doc_specific = "Give me a 5-point summary from the PDF"
print(f"\nQuery: {query_doc_specific}")
print(f"Response: {rag_answer(query_doc_specific)}")

print("\n" + "-" * 30 + "\n")

# Test 2: A general knowledge question (should trigger direct answer)
query_general = "What is machine learning?"
print(f"Query: {query_general}")
print(f"Response: {rag_answer(query_general)}")

print("\n" + "-" * 30 + "\n")

# Test 3: Another document-specific question
query_another_doc = "What are the main topics discussed in this book?"
print(f"Query: {query_another_doc}")
print(f"Response: {rag_answer(query_another_doc)}")


--- Testing RAG Pipeline ---

Query: Give me a 5-point summary from the PDF
🕵️ Agent decided to SEARCH document for: 'Give me a 5-point summary from the PDF'
Response: This book provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of models. It provides a comprehensive overview of the methods and tools used in the design of mode